In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

# ---- Config (paper defaults) ----
D_MODEL    = 1024   # LLM hidden dim; override per model (Qwen3-0.6B=1024, 1.7B=2048)
D_BOT      = 256    # VQ-VAE bottleneck dim (encoder output / codebook dim)
L          = 16     # compression rate: L text tokens → 1 latent token
C_SIZE     = 1024   # codebook size
BETA       = 0.25   # commitment loss weight

LR_VQVAE  = 1e-5   # paper: Adam lr=1e-5
BATCH_SIZE = 32     # paper: batch=32
STEPS      = 500    # demo steps (paper: 100k)

# Randomized replacement schedule (paper: M = multiples of L up to 256)
M_SET = [0, 72, 128, 160, 192, 224, 256]

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
print(f"L={L}, C_SIZE={C_SIZE}, D_model={D_MODEL}, D_bot={D_BOT}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


L=16, C_SIZE=1024, D_model=1024, D_bot=256


In [2]:
import sys
sys.path.insert(0, "/Users/fangyuanyu/Implementation/mod_gpt")

from sorl.tokenassort import (
    TokenAssortedVQVAE,
    sample_replacement_length,
    assign_latent_ids,
    build_mixed_sequence,
    add_abs_special_tokens,
    DEFAULT_L, DEFAULT_C_SIZE, DEFAULT_D_BOT, DEFAULT_BETA, DEFAULT_M_SET,
)

print("Imported from sorl.tokenassort")

Imported from sorl.tokenassort


In [3]:
# Baseline to compare against: TokenAssorted
# [26]. Token Assorted: Mixing Latent and Text Tokens for Improved Language Model Reasoning

# VQ-VAE Training For each benchmark, we train a VQVAE for 100k steps using the Adam optimizer, with learning rate  10−5 and batch size 32. We use a codebook of size 1024 and compress every chunk of L = 16 tokens into a single latent token (i.e., the compression rate r = 16).

# Randomized Latent Code Replacement We introduce a stochastic procedure for partially replacing CoT tokens with latent codes. Specifically, we define a set of predetermined numbers M = {0, 72, 128, 160, 192, 224, 256}, which are all multipliers of L = 16. For each training example, we first sample  mmax ∈ Mthen sample an integer m ∈ [0, 16, 32, . . . , mmax]uniformly at random. The first m CoT tokens are replaced by their corresponding latent discrete codes, while the later ones remain as raw text. This stochastic replacement mechanism exposes the model to a wide range of latent-text mixtures, enabling it to effectively learn from varying degrees of latent abstraction.
# Note: they put abstract tokens inside </abs_begin> ... </abs_end> braket, so the data processing function needs to be updated accordingly
#       I assume during evaluation time, they directly use the model to generate CoT with abstraction involved

In [3]:
# ---- (3) Full GSM8K pipeline ----
# Step A: load dataset, parse question / CoT / answer
# Step B: extract CoT embeddings from a frozen LLM
# Step C: train VQ-VAE on the extracted embeddings (→ labeler)
# Step D: for each training sample, call build_mixed_sequence → mixed_ids

from datasets import load_dataset

# --- A. Load GSM8K ---
gsm = load_dataset("gsm8k", "main")
train_ds = gsm["train"]   # 7473 samples; each has "question" and "answer"

def parse_gsm8k(sample):
    """Split GSM8K answer into CoT reasoning and final answer (after '####')."""
    answer_str = sample["answer"]
    if "####" in answer_str:
        cot_text, final = answer_str.split("####", 1)
        return sample["question"].strip(), cot_text.strip(), final.strip()
    return sample["question"].strip(), answer_str.strip(), ""

In [4]:
# ---- Step B: Extract CoT embeddings from a frozen LLM ----
# Replace the mock below with your actual model + tokenizer.
# The labeler only needs token embeddings (layer 0 = token embedding table),
# or any intermediate hidden state — layer 0 is cheapest and avoids loading
# the full model on a CPU notebook.

from transformers import AutoTokenizer, AutoModel
import torch

MODEL_ID = "Qwen/Qwen3-0.6B"

print(f"Loading tokenizer for {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# --- extract token embedding matrix (no forward pass needed) ---
print(f"Loading embedding table only (no full forward pass) ...")
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
emb_table = model.get_input_embeddings().weight.detach().float()  # cast bf16 → float32
D_MODEL_REAL = emb_table.shape[1]
print(f"Embedding table: {emb_table.shape}   dtype={emb_table.dtype}   D_model={D_MODEL_REAL}")

# Update config to match real D_MODEL
D_MODEL = D_MODEL_REAL

Loading tokenizer for Qwen/Qwen3-0.6B ...
Loading embedding table only (no full forward pass) ...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding table: torch.Size([151936, 1024])   dtype=torch.float32   D_model=1024


In [5]:
# ---- Step C prep: extract chunk token IDs from full text ----
# Store (n_chunks, L) int64 IDs instead of embeddings — only ~10 MB for all GSM8K.
# Embeddings are looked up on-the-fly during VQ-VAE training: emb_table[batch_ids].

def get_full_text_chunk_ids(sample, tokenizer, L=16):
    """FULL text (q + cot + ans) → (n_chunks, L) token-ID tensor."""
    full_text = sample["question"] + " " + sample["answer"]
    ids = tokenizer(full_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    n_chunks = len(ids) // L
    if n_chunks == 0:
        return None
    return ids[:n_chunks * L].reshape(n_chunks, L)   # (n_chunks, L)

N_SAMPLES_EMB = len(train_ds)
print(f"Extracting chunk token IDs from {N_SAMPLES_EMB} samples ...")
chunk_id_list = []
for i in range(N_SAMPLES_EMB):
    cids = get_full_text_chunk_ids(train_ds[i], tokenizer, L)
    if cids is not None:
        chunk_id_list.append(cids)

all_chunk_ids = torch.cat(chunk_id_list, dim=0)   # (N_total, L)  int64
print(f"Total chunks : {all_chunk_ids.shape[0]}  shape: {all_chunk_ids.shape}")
print(f"Memory (IDs) : {all_chunk_ids.nbytes / 1024**2:.1f} MB  (vs {all_chunk_ids.shape[0] * L * D_MODEL * 4 / 1024**2:.0f} MB if stored as float32)")

Extracting chunk token IDs from 7473 samples ...
Total chunks : 81221  shape: torch.Size([81221, 16])
Memory (IDs) : 9.9 MB  (vs 5076 MB if stored as float32)


In [6]:
# ---- Step C: Train TokenAssortedVQVAE (EMA codebook + dead-code reinit) ----
# Encoder: (B, L, D) → flatten → nn.Linear(L*D, D_bot) → (B, D_bot)
# VQ:      EMA codebook update + dead-code reinitialization (standard VQ-VAE-2 trick)
# Decoder: (B, D_bot) → nn.Linear(D_bot, L*D) → (B, L, D)
#
# Codebook collapse fix:
#   - Codebook entries move via EMA toward their assigned encoder outputs (no gradient).
#   - Entries with EMA usage < dead_threshold are reborn from random batch elements.
#   - Only the encoder sees gradients (commitment loss: beta * ||sg(e_k) - z||^2).
#
# Monitoring (from MishaLaskin/vqvae reference):
#   perplexity = exp(entropy of batch assignment dist); max = C_SIZE (fully uniform).

from sorl.tokenassort import TokenAssortedVQVAE

torch.manual_seed(SEED)
vqvae = TokenAssortedVQVAE(D_MODEL, L=L, D_bot=D_BOT, C_SIZE=C_SIZE, beta=BETA,
                            decay=0.99, dead_threshold=0.01)
# Only encoder + decoder parameters need optimizer — codebook updated via EMA.
enc_dec_params = list(vqvae.encoder.parameters()) + list(vqvae.decoder.parameters())
opt = torch.optim.Adam(enc_dec_params, lr=LR_VQVAE)

STEPS_VQVAE = 20000
print(f"Training TokenAssortedVQVAE (EMA codebook, dead_threshold=0.01) for {STEPS_VQVAE} steps ...")
print(f"  encoder: ({L}×{D_MODEL}) → D_bot={D_BOT} → codebook {C_SIZE}  (max perplexity={C_SIZE})")
print(f"{'step':>6}  {'recon':>8}  {'commit':>8}  {'total':>8}  {'perplexity':>12}  {'vocab_util':>10}")
print("-" * 68)

for step in range(STEPS_VQVAE):
    idx = torch.randperm(len(all_chunk_ids))[:BATCH_SIZE]
    x_b = emb_table[all_chunk_ids[idx]]           # (B, L, D)  — on-the-fly lookup
    ids_out, x_hat, recon_loss, commit_loss, total_loss = vqvae(x_b)
    opt.zero_grad(); total_loss.backward(); opt.step()

    if (step + 1) % 200 == 0:
        with torch.no_grad():
            s_idx = torch.randperm(len(all_chunk_ids))[:2048]
            x_s   = emb_table[all_chunk_ids[s_idx]]
            
            # Vocab util over larger sample
            util = vqvae.vocab_utilization(x_s)
            
            # Perplexity over training batch
            e_mean = F.one_hot(ids_out, C_SIZE).float().mean(0)           # (C,)
            perplexity = torch.exp(-(e_mean * (e_mean + 1e-10).log()).sum()).item()
            
        print(f"{step+1:>6}  {recon_loss.item():>8.4f}  {commit_loss.item():>8.4f}  {total_loss.item():>8.4f}  {perplexity:>12.1f}  {util:>10.3f}")

print("\nvqvae is now the chunk labeler for GSM8K.")

Training TokenAssortedVQVAE (EMA codebook, dead_threshold=0.01) for 20000 steps ...
  encoder: (16×1024) → D_bot=256 → codebook 1024  (max perplexity=1024)
  step     recon    commit     total    perplexity  vocab_util
--------------------------------------------------------------------
   200    0.0019    0.0000    0.0019           3.2       0.018
   400    0.0017    0.0000    0.0017           4.4       0.014
   600    0.0014    0.0000    0.0014           4.0       0.016
   800    0.0010    0.0000    0.0010           4.9       0.017
  1000    0.0008    0.0000    0.0009           6.1       0.019
  1200    0.0007    0.0000    0.0008          10.4       0.029
  1400    0.0007    0.0000    0.0007           8.6       0.032
  1600    0.0007    0.0000    0.0007          14.8       0.035
  1800    0.0006    0.0000    0.0006          14.4       0.037
  2000    0.0007    0.0000    0.0007          14.4       0.035
  2200    0.0006    0.0000    0.0007          15.0       0.042
  2400    0.0007   

In [8]:
# ---- Step D: Build mixed sequences using trained vqvae ----

from sorl.tokenassort import add_abs_special_tokens, sample_replacement_length

abs_begin_id, abs_end_id = add_abs_special_tokens(tokenizer)
LATENT_OFFSET = len(tokenizer)

def get_cot_chunks(cot_text, tokenizer, emb_table, L=16):
    """CoT text → (n_chunks, L, D_MODEL) raw chunk tensor for labeling."""
    ids = tokenizer(cot_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    n_chunks = len(ids) // L
    if n_chunks == 0:
        return torch.zeros(0, L, emb_table.shape[1])
    ids_trunc = ids[:n_chunks * L].reshape(n_chunks, L)
    return emb_table[ids_trunc]   # (n_chunks, L, D)

def prepare_sample(sample, tokenizer, emb_table, vqvae, latent_offset,
                   abs_begin_id=None, abs_end_id=None, L=16, M_set=None):
    """Full pipeline: raw (n_chunks, L, D) chunks → vqvae.encode() → mixed sequence."""
    q_text, cot_text, ans_text = parse_gsm8k(sample)
    q_ids   = tokenizer(q_text,   add_special_tokens=False)["input_ids"]
    cot_ids = tokenizer(cot_text, add_special_tokens=False)["input_ids"]
    ans_ids = tokenizer(ans_text, add_special_tokens=False)["input_ids"]

    cot_chunks = get_cot_chunks(cot_text, tokenizer, emb_table, L)
    n_chunks   = cot_chunks.shape[0]

    m         = sample_replacement_length(len(cot_ids), M_set=M_set, L=L)
    n_replace = m // L

    mixed = list(q_ids)
    if n_replace > 0 and n_chunks > 0:
        if abs_begin_id is not None:
            mixed.append(abs_begin_id)
        with torch.no_grad():
            lat_ids = vqvae.encode(cot_chunks)   # (n_chunks,)
        for k in range(min(n_replace, n_chunks)):
            mixed.append(int(lat_ids[k].item()) + latent_offset)
        if abs_end_id is not None:
            mixed.append(abs_end_id)

    mixed += list(cot_ids[m:])
    if ans_ids:
        mixed += list(ans_ids)
    return mixed, m

print(f"abs_begin_id  : {abs_begin_id}  ({tokenizer.convert_ids_to_tokens(abs_begin_id)})")
print(f"abs_end_id    : {abs_end_id}  ({tokenizer.convert_ids_to_tokens(abs_end_id)})")
print(f"Latent range  : [{LATENT_OFFSET}, {LATENT_OFFSET + C_SIZE})")
print(f"\n{'idx':>4}  {'m':>4}  {'n_lat':>6}  {'n_txt_cot':>9}  {'total_len':>9}")
print("-" * 42)
for i in range(8):
    mixed, m = prepare_sample(train_ds[i], tokenizer, emb_table, vqvae,
                               LATENT_OFFSET, abs_begin_id, abs_end_id)
    _, cot_i, _ = parse_gsm8k(train_ds[i])
    T_cot = len(tokenizer(cot_i, add_special_tokens=False)["input_ids"])
    print(f"{i:>4}  {m:>4}  {m//L:>6}  {T_cot - m:>9}  {len(mixed):>9}")

abs_begin_id  : 151669  (<abs_begin>)
abs_end_id    : 151670  (<abs_end>)
Latent range  : [151671, 152695)

 idx     m   n_lat  n_txt_cot  total_len
------------------------------------------
   0     0       0         55         96
   1     0       0         60         93
   2    16       1         81        146
   3     0       0        122        178
   4    64       4          7         42
   5     0       0        158        225
   6     0       0         83        142
   7     0       0        110        218


In [ ]:
# ---- Stacked Abstraction via Representation Engineering ----
# Instead of interleaving latent codes into the token sequence (TokenAssorted),
# keep the full NL sequence intact. The VQ-VAE assigns each L-token chunk a
# discrete code k. A *separate learnable* steering embedding table maps
# k → steering_vector(D_MODEL). At selected transformer layer(s), the steering
# vector is added to hidden states at positions belonging to that chunk.
#
# The model sees the full text, but its internal representations are "steered"
# by per-chunk abstract codes. The steering embeddings are trained end-to-end
# with the causal LM loss.

# Step 1: Pre-compute chunk → VQ code assignments for all CoT spans

import torch

@torch.no_grad()
def precompute_chunk_codes(dataset, tokenizer, emb_table, vqvae, L=16):
    """
    For each sample, tokenize full text, chunk the CoT portion,
    and assign VQ codes. Returns list of dicts:
      {
        'input_ids': list[int],       # full NL token ids (q + cot + ans)
        'prompt_len': int,            # length of question tokens
        'chunk_codes': list[int],     # VQ code per L-token chunk of CoT
        'cot_start': int,             # position where CoT starts in input_ids
        'cot_len': int,               # number of CoT tokens
      }
    """
    records = []
    for i in range(len(dataset)):
        sample = dataset[i]
        answer_str = sample["answer"]
        q_text = sample["question"].strip()
        if "####" in answer_str:
            cot_text, final = answer_str.split("####", 1)
            cot_text, final = cot_text.strip(), final.strip()
        else:
            cot_text, final = answer_str.strip(), ""

        q_ids   = tokenizer(q_text,   add_special_tokens=False)["input_ids"]
        cot_ids = tokenizer(cot_text, add_special_tokens=False)["input_ids"]
        ans_ids = tokenizer(final,    add_special_tokens=False)["input_ids"]

        full_ids = q_ids + cot_ids + ans_ids
        cot_start = len(q_ids)
        cot_len   = len(cot_ids)
        n_chunks  = cot_len // L

        if n_chunks > 0:
            chunk_token_ids = torch.tensor(cot_ids[:n_chunks * L]).reshape(n_chunks, L)
            chunk_embs = emb_table[chunk_token_ids]          # (n_chunks, L, D)
            codes = vqvae.encode(chunk_embs).tolist()        # list[int]
        else:
            codes = []

        records.append({
            'input_ids':   full_ids,
            'prompt_len':  len(q_ids),
            'chunk_codes': codes,
            'cot_start':   cot_start,
            'cot_len':     cot_len,
        })
    return records

print("Pre-computing chunk codes for all GSM8K train samples ...")
records = precompute_chunk_codes(train_ds, tokenizer, emb_table, vqvae, L=L)

# Stats
n_with_codes = sum(1 for r in records if len(r['chunk_codes']) > 0)
total_chunks = sum(len(r['chunk_codes']) for r in records)
print(f"Samples with ≥1 chunk: {n_with_codes}/{len(records)}")
print(f"Total chunks assigned: {total_chunks}")
print(f"Example record[0] codes[:8]: {records[0]['chunk_codes'][:8]}")

In [ ]:
# ---- Step 2: StackedAbstractionWrapper ----
# Wraps a causal LM. Registers forward hooks on selected layers to add
# per-chunk steering vectors to hidden states.

import torch
import torch.nn as nn

class StackedAbstractionWrapper(nn.Module):
    """
    Wraps a HuggingFace causal LM and injects learnable steering vectors
    into selected transformer layers based on pre-computed VQ chunk codes.
    
    For each L-token chunk assigned code k, steering_emb[k] is added to
    the hidden states of those L positions at the hooked layer(s).
    
    Usage:
        wrapper = StackedAbstractionWrapper(model, C_SIZE, D_MODEL, inject_layers=[12])
        loss = wrapper(input_ids, attention_mask, labels, chunk_codes, cot_start)
    """
    def __init__(self, model, C_SIZE, D_MODEL, inject_layers=None, 
                 scale=0.1, L=16):
        super().__init__()
        self.model = model
        self.L = L
        self.scale = scale
        
        # Learnable steering embeddings: one vector per VQ code
        self.steering_emb = nn.Embedding(C_SIZE, D_MODEL)
        nn.init.zeros_(self.steering_emb.weight)  # start with no steering
        
        # Which layers to inject into (0-indexed transformer layers)
        n_layers = model.config.num_hidden_layers
        if inject_layers is None:
            # Default: inject at the middle layer
            inject_layers = [n_layers // 2]
        self.inject_layers = inject_layers
        
        # State set per forward pass (used by hooks)
        self._steering_map = None   # (seq_len,) tensor of code indices, -1 = no steering
        self._hooks = []
        self._register_hooks()
    
    def _register_hooks(self):
        """Register forward hooks on selected transformer layers."""
        for hook in self._hooks:
            hook.remove()
        self._hooks = []
        
        for layer_idx in self.inject_layers:
            layer = self.model.model.layers[layer_idx]
            hook = layer.register_forward_hook(self._steering_hook)
            self._hooks.append(hook)
    
    def _steering_hook(self, module, input, output):
        """Add steering vectors to the hidden states output of a transformer layer."""
        if self._steering_map is None:
            return output
        
        # output is a tuple: (hidden_states, ...) for HF transformers
        hidden_states = output[0]  # (B, S, D)
        B, S, D = hidden_states.shape
        
        # _steering_map: (B, S) with code indices; -1 = no steering
        steer_map = self._steering_map  # (B, S)
        mask = steer_map >= 0           # (B, S) bool
        
        if mask.any():
            # Gather steering vectors for all valid positions
            safe_codes = steer_map.clamp(min=0)  # replace -1 with 0 for indexing
            steer_vecs = self.steering_emb(safe_codes)  # (B, S, D)
            steer_vecs = steer_vecs * mask.unsqueeze(-1).float() * self.scale
            
            # Cast to match hidden states dtype
            hidden_states = hidden_states + steer_vecs.to(hidden_states.dtype)
        
        # Return modified output tuple
        return (hidden_states,) + output[1:]
    
    def _build_steering_map(self, batch_size, seq_len, chunk_codes_list, 
                            cot_starts, device):
        """
        Build (B, S) tensor mapping each position to its VQ code index.
        Positions without steering get -1.
        """
        steer_map = torch.full((batch_size, seq_len), -1, dtype=torch.long, device=device)
        
        for b in range(batch_size):
            codes = chunk_codes_list[b]
            cot_s = cot_starts[b]
            for c_idx, code in enumerate(codes):
                pos_start = cot_s + c_idx * self.L
                pos_end   = min(pos_start + self.L, seq_len)
                if pos_start < seq_len:
                    steer_map[b, pos_start:pos_end] = code
        return steer_map
    
    def forward(self, input_ids, attention_mask, labels, 
                chunk_codes_list, cot_starts):
        """
        Args:
            input_ids:        (B, S) token ids (full NL sequence, no latent codes)
            attention_mask:   (B, S)
            labels:           (B, S) with -100 for prompt/padding
            chunk_codes_list: list[list[int]] — VQ codes per chunk, per sample
            cot_starts:       list[int] — position where CoT starts per sample
        """
        B, S = input_ids.shape
        self._steering_map = self._build_steering_map(
            B, S, chunk_codes_list, cot_starts, input_ids.device
        )
        
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        
        self._steering_map = None  # clean up
        return outputs
    
    def remove_hooks(self):
        for hook in self._hooks:
            hook.remove()
        self._hooks = []


print(f"StackedAbstractionWrapper defined.")
print(f"  - Learnable steering_emb: ({C_SIZE}, {D_MODEL})")
print(f"  - Injection scale: 0.1 (init zeros → no disturbance at start)")


--- trial 0  m=48  (3 latent tokens replacing 48 text tokens) ---
'W' 'eng' ' earns' ' $' '1' '2' ' an' ' hour' ' for' ' babys' 'itting' '.' ' Yesterday' ',' ' she' ' just' ' did' ' ' '5' '0' ' minutes' ' of' ' babys' 'itting' '.' ' How' ' much' ' did' ' she' ' earn' '?' <abs_begin> [LAT:170] [LAT:139] [LAT:137] <abs_end> '.' '2' '*' '5' '0' '=' '1' '0' '>>' '1' '0' '.' '1' '0' 

--- trial 1  m=48  (3 latent tokens replacing 48 text tokens) ---
'W' 'eng' ' earns' ' $' '1' '2' ' an' ' hour' ' for' ' babys' 'itting' '.' ' Yesterday' ',' ' she' ' just' ' did' ' ' '5' '0' ' minutes' ' of' ' babys' 'itting' '.' ' How' ' much' ' did' ' she' ' earn' '?' <abs_begin> [LAT:170] [LAT:139] [LAT:137] <abs_end> '.' '2' '*' '5' '0' '=' '1' '0' '>>' '1' '0' '.' '1' '0' 

--- trial 2  m=0  (0 latent tokens replacing 0 text tokens) ---
'W' 'eng' ' earns' ' $' '1' '2' ' an' ' hour' ' for' ' babys' 'itting' '.' ' Yesterday' ',' ' she' ' just' ' did' ' ' '5' '0' ' minutes' ' of' ' babys' 'itting' '.' ' Ho

In [ ]:
# ---- Step 3: Dataset + Collate for Stacked Abstraction ----

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class StackedAbstractionDataset(Dataset):
    """Each sample: full NL input_ids + chunk VQ codes + positions."""
    def __init__(self, records):
        self.records = records
    
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            'input_ids':   torch.tensor(r['input_ids'], dtype=torch.long),
            'prompt_len':  r['prompt_len'],
            'chunk_codes': r['chunk_codes'],   # list[int]
            'cot_start':   r['cot_start'],
        }


def stacked_collate_fn(batch, pad_token_id):
    """Pad input_ids, build labels, collect chunk_codes & cot_starts."""
    input_ids   = [item['input_ids'] for item in batch]
    prompt_lens = [item['prompt_len'] for item in batch]
    chunk_codes = [item['chunk_codes'] for item in batch]
    cot_starts  = [item['cot_start'] for item in batch]
    
    padded = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    attn_mask = (padded != pad_token_id).long()
    
    labels = padded.clone()
    labels[labels == pad_token_id] = -100
    for i, p_len in enumerate(prompt_lens):
        labels[i, :p_len] = -100
    
    return {
        'input_ids':        padded,
        'attention_mask':   attn_mask,
        'labels':           labels,
        'chunk_codes_list': chunk_codes,
        'cot_starts':       cot_starts,
    }


stacked_ds = StackedAbstractionDataset(records)
pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

stacked_loader = DataLoader(
    stacked_ds, batch_size=4, shuffle=True,
    collate_fn=lambda b: stacked_collate_fn(b, pad_id)
)

# Quick sanity check
batch = next(iter(stacked_loader))
print(f"input_ids      : {batch['input_ids'].shape}")
print(f"labels         : {batch['labels'].shape}")
print(f"chunk_codes[0] : {batch['chunk_codes_list'][0][:6]} ...")
print(f"cot_starts     : {batch['cot_starts']}")

input_ids shape      : torch.Size([2, 180])
attention_mask shape : torch.Size([2, 180])
labels shape         : torch.Size([2, 180])

Example row 0 labels (first 40 tokens, -100 means ignored):
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [ ]:
# ---- Step 4: Training Loop — Stacked Abstraction ----
# Compare two settings:
#   (A) Baseline: standard SFT on full NL sequences (no steering)
#   (B) Stacked:  same sequences, but with learned per-chunk steering vectors

from transformers import AutoModelForCausalLM

# --- Load fresh model ---
print(f"Loading {MODEL_ID} ...")
sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

# --- Wrap with stacked abstraction ---
n_layers = sft_model.config.num_hidden_layers
inject_at = [n_layers // 2]  # middle layer
print(f"Model has {n_layers} layers, injecting steering at layer(s) {inject_at}")

wrapper = StackedAbstractionWrapper(
    sft_model, C_SIZE=C_SIZE, D_MODEL=D_MODEL,
    inject_layers=inject_at, scale=0.1, L=L
)

# Move steering embeddings to same device/dtype as model
device = sft_model.device
wrapper.steering_emb = wrapper.steering_emb.to(device=device, dtype=torch.float32)

# --- Optimizer: LM params + steering embeddings ---
# Use higher lr for steering embeddings since they start at zero
optimizer = torch.optim.AdamW([
    {'params': sft_model.parameters(), 'lr': 2e-5},
    {'params': wrapper.steering_emb.parameters(), 'lr': 1e-3},
])

sft_model.train()
N_STEPS = 30

print(f"\nTraining stacked abstraction for {N_STEPS} steps ...")
print(f"{'step':>4}  {'loss':>8}  {'n_steered_pos':>14}")
print("-" * 32)

for step, batch in enumerate(stacked_loader):
    if step >= N_STEPS:
        break
    
    input_ids = batch['input_ids'].to(device)
    attn_mask = batch['attention_mask'].to(device)
    labels    = batch['labels'].to(device)
    
    outputs = wrapper(
        input_ids, attn_mask, labels,
        chunk_codes_list=batch['chunk_codes_list'],
        cot_starts=batch['cot_starts'],
    )
    
    loss = outputs.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Count how many positions got steered in this batch
    n_steered = sum(len(c) * L for c in batch['chunk_codes_list'])
    
    if (step + 1) % 5 == 0 or step == 0:
        print(f"{step+1:>4}  {loss.item():>8.4f}  {n_steered:>14}")

wrapper.remove_hooks()
print("\nDone. Steering embeddings learned end-to-end with causal LM loss.")

Loading Qwen/Qwen3-0.6B for causal LM training...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Resizing model embeddings from 151936 -> 151671

Starting minimal SFT loop for 10 steps...
Step  1/10 | Loss: 4.4803
Step  2/10 | Loss: 1.9531
Step  3/10 | Loss: 2.7169
Step  4/10 | Loss: 2.9767
Step  5/10 | Loss: 1.3342
Step  6/10 | Loss: 1.9172
Step  7/10 | Loss: 2.3034
Step  8/10 | Loss: 3.0192
Step  9/10 | Loss: 1.6165
Step 10/10 | Loss: 1.4972

Success! The mixed sequences are fully compatible with standard causal LM training.


In [ ]:
# ---- Step 5: Inspect learned steering embeddings ----

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with torch.no_grad():
    steer_w = wrapper.steering_emb.weight.float().cpu()  # (C_SIZE, D)
    norms = steer_w.norm(dim=-1)                          # (C_SIZE,)

    # Which codes were actually used in training?
    used_codes = set()
    for r in records:
        used_codes.update(r['chunk_codes'])
    used_mask = torch.tensor([i in used_codes for i in range(C_SIZE)])
    
    print(f"Codebook size     : {C_SIZE}")
    print(f"Codes used in data: {len(used_codes)}")
    print(f"Steering norm (used codes)   mean={norms[used_mask].mean():.4f}  max={norms[used_mask].max():.4f}")
    print(f"Steering norm (unused codes) mean={norms[~used_mask].mean():.6f}")

    # Cosine similarity between top steering vectors
    top_k = 20
    top_idx = norms.topk(top_k).indices
    top_vecs = steer_w[top_idx]
    top_vecs_normed = top_vecs / (top_vecs.norm(dim=-1, keepdim=True) + 1e-8)
    cos_sim = top_vecs_normed @ top_vecs_normed.T
    
    print(f"\nTop-{top_k} steering vectors by norm:")
    for i, idx in enumerate(top_idx[:10]):
        print(f"  code {idx.item():>4}  norm={norms[idx]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of steering norms
axes[0].hist(norms[used_mask].numpy(), bins=50, alpha=0.7, label='used codes')
axes[0].hist(norms[~used_mask].numpy(), bins=50, alpha=0.5, label='unused codes')
axes[0].set_xlabel('Steering vector L2 norm')
axes[0].set_ylabel('Count')
axes[0].set_title('Steering Embedding Norms')
axes[0].legend()

# Cosine sim heatmap of top-k
im = axes[1].imshow(cos_sim.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title(f'Cosine Sim (top-{top_k} by norm)')
axes[1].set_xlabel('Code index')
axes[1].set_ylabel('Code index')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig('figure/stacked_abstraction_steering.png', dpi=150)
plt.show()
print("Saved to figure/stacked_abstraction_steering.png")